# 01 · LangChain basics: tools, tool calls and the agent loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/agent-frameworks/01_langchain_basics.ipynb) &nbsp; 📖 Concept deck: [LangChain basics](https://nilesh-salpe.github.io/visual-ai/agent-frameworks/01-langchain-basics/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** a tool with a generated JSON schema, an executor that turns tool calls into tool messages (including errors), and the agent loop by hand, then the same agent with `create_agent`.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with a scripted fake chat model, so no API key or GPU is needed. Run the **Install** cell first (pinned to LangChain 1.4.6 and LangGraph 1.2.15).

In [ ]:
# @title Install pinned packages
%pip install -q "langchain==1.4.6" "langchain-core==1.6.10" "langgraph==1.2.15"

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool

class ScriptedChatModel(BaseChatModel):
    """A stand-in for a real chat model: replies with the scripted messages in order (cycling).
    Swap in e.g. init_chat_model("anthropic:claude-sonnet-5-5") to use a real model."""
    script: list
    i: int = 0
    calls: int = 0
    @property
    def _llm_type(self):
        return "scripted"
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        msg = self.script[self.i % len(self.script)]
        self.i += 1; self.calls += 1
        return ChatResult(generations=[ChatGeneration(message=msg)])
    def bind_tools(self, tools, **kwargs):
        return self          # a real model would attach the tool schemas to every request

def call(name, args, id):
    """Shorthand for one tool call inside an AIMessage."""
    return {"name": name, "args": args, "id": id}

@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    return f"18°C and sunny in {city}"

## Task 1: A tool with a schema

Return a LangChain tool named `convert_currency(amount: float, rate: float) -> float` that multiplies the two numbers. Use the `@tool` decorator: it reads the function name, the **docstring** (which becomes the description the model sees) and the **type hints** (which become the JSON schema).

In [ ]:
def build_currency_tool():
    # TODO: define convert_currency with @tool, a docstring and type hints, and return it
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_build_currency_tool(f):
    t = f()
    assert t.name == "convert_currency", f"tool name is {t.name!r}"
    assert t.description, "add a docstring: it is the description the model reads"
    schema = t.tool_call_schema.model_json_schema()
    assert sorted(schema["required"]) == ["amount", "rate"], f"required args: {schema['required']}"
    assert schema["properties"]["amount"]["type"] == "number", "type-hint amount as float"
    assert t.invoke({"amount": 10, "rate": 0.5}) == 5.0

check("build_currency_tool", _test_build_currency_tool, build_currency_tool)

## Task 2: Execute tool calls

Given an `AIMessage` whose `tool_calls` is a list of `{"name", "args", "id"}` dicts, run each call and return a list of `ToolMessage`s, one per call, **in order**, each with `tool_call_id` set to the call's id.
* Unknown tool → `ToolMessage(content="Error: unknown tool <name>", status="error", ...)`.
* A tool that raises → `ToolMessage(content="Error: <exception text>", status="error", ...)`. Returning errors lets the model fix its call instead of crashing the loop.

In [ ]:
def run_tool_calls(ai_message, tools):
    by_name = {t.name: t for t in tools}
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_run_tool_calls(f):
    @tool
    def divide(a: float, b: float) -> float:
        """Divide a by b."""
        return a / b
    ai = AIMessage(content="", tool_calls=[call("get_weather", {"city": "Paris"}, "t1"), call("divide", {"a": 1, "b": 0}, "t2"), call("nope", {}, "t3")])
    out = f(ai, [get_weather, divide])
    assert [m.tool_call_id for m in out] == ["t1", "t2", "t3"], "one ToolMessage per call, in order"
    assert all(isinstance(m, ToolMessage) for m in out)
    assert out[0].content == "18°C and sunny in Paris" and out[0].status == "success"
    assert out[1].status == "error" and "division" in out[1].content, "return the exception text as an error result"
    assert out[2].status == "error" and "unknown tool" in out[2].content

check("run_tool_calls", _test_run_tool_calls, run_tool_calls)

## Task 3: The agent loop by hand

Write the loop from the "What an agent is" deck: start with `[HumanMessage(question)]`, call `model.invoke(messages)`, append the reply; if it has no `tool_calls`, return the messages; otherwise append the tool results (use your `run_tool_calls`) and go again. Raise `RuntimeError` after `max_steps` model calls without a final answer.

In [ ]:
def agent_loop(model, tools, question, max_steps=5):
    model = model.bind_tools(tools)
    messages = [HumanMessage(question)]
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_agent_loop(f):
    m = ScriptedChatModel(script=[AIMessage(content="", tool_calls=[call("get_weather", {"city": "Oslo"}, "a")]), AIMessage(content="Oslo: 18°C.")])
    msgs = f(m, [get_weather], "Weather in Oslo?")
    kinds = [type(x).__name__ for x in msgs]
    assert kinds == ["HumanMessage", "AIMessage", "ToolMessage", "AIMessage"], kinds
    assert msgs[2].content == "18°C and sunny in Oslo" and msgs[-1].content == "Oslo: 18°C."
    looping = ScriptedChatModel(script=[AIMessage(content="", tool_calls=[call("get_weather", {"city": "X"}, "x")])])
    try:
        f(looping, [get_weather], "?", max_steps=3)
        assert False, "a model that never stops must hit the step limit"
    except RuntimeError:
        pass
    assert looping.calls == 3, f"expected exactly 3 model calls, got {looping.calls}"

check("agent_loop", _test_agent_loop, agent_loop)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: A tool with a schema
def build_currency_tool_solution():
    @tool
    def convert_currency(amount: float, rate: float) -> float:
        """Convert an amount of money with an exchange rate."""
        return amount * rate
    return convert_currency

check("build_currency_tool (reference)", _test_build_currency_tool, build_currency_tool_solution)

In [ ]:
# @title Solution for Task 2: Execute tool calls
def run_tool_calls_solution(ai_message, tools):
    by_name = {t.name: t for t in tools}
    out = []
    for tc in ai_message.tool_calls:
        t = by_name.get(tc["name"])
        if t is None:
            out.append(ToolMessage(content=f"Error: unknown tool {tc['name']}", tool_call_id=tc["id"], status="error"))
            continue
        try:
            out.append(ToolMessage(content=str(t.invoke(tc["args"])), tool_call_id=tc["id"], name=tc["name"]))
        except Exception as e:
            out.append(ToolMessage(content=f"Error: {e}", tool_call_id=tc["id"], status="error"))
    return out

check("run_tool_calls (reference)", _test_run_tool_calls, run_tool_calls_solution)

In [ ]:
# @title Solution for Task 3: The agent loop by hand
def agent_loop_solution(model, tools, question, max_steps=5):
    model = model.bind_tools(tools)
    messages = [HumanMessage(question)]
    for _ in range(max_steps):
        ai = model.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return messages
        messages += run_tool_calls_solution(ai, tools)
    raise RuntimeError("step limit reached")

check("agent_loop (reference)", _test_agent_loop, agent_loop_solution)

---
## Experiments
The same agent with `create_agent`: it compiles a LangGraph graph with a `model` node and a `tools` node and runs the loop you just wrote.

In [ ]:
from langchain.agents import create_agent
m = ScriptedChatModel(script=[AIMessage(content="", tool_calls=[call("get_weather", {"city": "Paris"}, "t1")]), AIMessage(content="It is 18°C in Paris.")])
agent = create_agent(m, [get_weather], system_prompt="You are a helpful assistant.")
out = agent.invoke({"messages": [HumanMessage("Weather in Paris?")]})
for msg in out["messages"]:
    print(f"{type(msg).__name__:13}", msg.content or msg.tool_calls)
print("graph nodes:", list(agent.get_graph().nodes))

---
## Takeaways
* A tool is a function plus a **schema and description** the model reads; the docstring is part of your prompt.
* The model never runs code: it returns structured **tool calls**; your code executes them and returns **tool messages** tied by id.
* `create_agent` is the same loop, compiled as a LangGraph graph, with middleware hooks around model and tool calls.